# Speech Emotion Recognition: Exploratory Data Analysis & Baseline

This notebook inspects the speech emotion datasets (**RAVDESS**, **TESS**, **EMO-DB**), visualizes acoustic properties (durations, waveforms, MFCC spectrograms), and benchmarks classical baselines against deep learning models (**CNN**, **BiLSTM**, **CNN-BiLSTM**).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import AudioConfig, PathConfig
from src.datasets import build_metadata_dataframe, split_dataset
from src.features import extract_mfcc_sequence, extract_pooled_features, load_and_preprocess_audio

sns.set_theme(style='whitegrid')
audio_cfg = AudioConfig()
path_cfg = PathConfig()

### 1. Discover Available Datasets & Validate Schema
We scan `data/raw/` (falling back to `data/demo/` if real datasets are not yet downloaded) and check for missing values, duplicate filepaths, and class distribution.

In [ ]:
raw_dir = path_cfg.data_dir if any(path_cfg.data_dir.glob('**/*.wav')) else path_cfg.raw_dir
df = build_metadata_dataframe(raw_dir, datasets=('ravdess', 'tess', 'emodb'))
print(f'Scanning: {raw_dir}')
print(f'Total valid records: {len(df)}')
print('Null counts:\n', df.isnull().sum())
df.head()

**Observation:** Inspect the printed null counts and sample rows above. Missing or unparseable filenames are filtered during metadata construction so every remaining row has a valid `filepath`, `dataset`, `speaker_id`, and canonical `emotion` label. Next, we visualize the class and dataset distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
order = df['emotion'].value_counts().index
sns.countplot(data=df, x='emotion', order=order, ax=axes[0], palette='viridis')
axes[0].set_title('Emotion Class Distribution')
axes[0].tick_params(axis='x', rotation=30)

sns.countplot(data=df, x='dataset', hue='emotion', ax=axes[1])
axes[1].set_title('Samples by Dataset and Emotion')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Analysis:** Note any class imbalance (for example, RAVDESS has twice as many `neutral` clips when `calm` is merged into `neutral`, whereas other emotions appear equally). In `src/train.py`, we handle this with `class_weight='balanced'` in `CrossEntropyLoss` and track **Macro F1** alongside **Accuracy**.

In [ ]:
sample_df = df.sample(n=min(300, len(df)), random_state=42)
durations = []
corrupt_files = 0
for fp in sample_df['filepath']:
    try:
        durations.append(librosa.get_duration(path=fp))
    except Exception:
        corrupt_files += 1

print(f'Corrupt/unreadable files in sample: {corrupt_files}')
print(pd.Series(durations).describe())

plt.figure(figsize=(8, 3.5))
sns.histplot(durations, bins=25, kde=True, color='#2563eb')
plt.axvline(audio_cfg.duration, color='red', linestyle='--', label=f'Target duration ({audio_cfg.duration}s)')
plt.title('Audio Clip Duration Distribution')
plt.xlabel('Duration (seconds)')
plt.legend()
plt.show()

**Analysis:** Speech clips across RAVDESS, TESS, and EMO-DB typically fall between 1.5s and 4.5s. Trimming leading/trailing silence and standardizing every clip to `3.0s` (`22050 Hz`) aligns temporal tensor shapes for batching while preserving the vocal utterance core.

In [ ]:
emotions_to_show = [e for e in ['angry', 'happy', 'sad', 'neutral'] if e in df['emotion'].unique()][:4]
fig, axes = plt.subplots(len(emotions_to_show), 2, figsize=(13, 3 * len(emotions_to_show)))
if len(emotions_to_show) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, emo in enumerate(emotions_to_show):
    fp = df[df['emotion'] == emo].iloc[0]['filepath']
    sig = load_and_preprocess_audio(fp, audio_cfg)
    mfcc_seq = extract_mfcc_sequence(sig, audio_cfg)

    axes[row_idx, 0].plot(sig, linewidth=0.7)
    axes[row_idx, 0].set_title(f'Waveform: {emo}')
    librosa.display.specshow(mfcc_seq[:, :audio_cfg.n_mfcc].T, sr=audio_cfg.sample_rate, x_axis='time', ax=axes[row_idx, 1], cmap='magma')
    axes[row_idx, 1].set_title(f'MFCCs (40 bands): {emo}')

plt.tight_layout()
plt.show()

**Analysis:** Compare the waveform envelopes and MFCC energy trajectories across emotions. High-arousal emotions (`angry`, `happy`) typically exhibit sharper amplitude bursts and richer high-frequency cepstral variations than low-arousal emotions (`sad`, `neutral`).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Strict split BEFORE fitting any scaler or encoder
train_df, val_df, test_df = split_dataset(df, speaker_independent=True, random_seed=42)

def featurize_pooled(sub_df):
    X = [extract_pooled_features(load_and_preprocess_audio(fp, audio_cfg), audio_cfg) for fp in sub_df['filepath']]
    return np.stack(X), sub_df['emotion'].tolist()

X_train, y_train_str = featurize_pooled(train_df)
X_test, y_test_str = featurize_pooled(test_df)

le = LabelEncoder()
y_train = le.fit_transform(y_train_str)
y_test = le.transform(y_test_str)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_sc, y_train)
preds = lr.predict(X_test_sc)

print(f'Baseline Accuracy: {accuracy_score(y_test, preds):.4f}')
print(f'Baseline Macro F1: {f1_score(y_test, preds, average="macro", zero_division=0):.4f}')
print(classification_report(y_test, preds, target_names=le.classes_, zero_division=0))

### Conclusion & Next Steps
- The classical Logistic Regression baseline on acoustic statistics provides a solid benchmark while respecting speaker-independent splitting and strict train-only scaling.
- Run `python -m src.train --model mlp` or `python -m src.train --model ensemble` to train deep acoustic neural networks, or `python -m src.train --model cnn_lstm` for PyTorch hybrid CNN-BiLSTM.